In [1]:
import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from datetime import datetime

In [3]:
df = pl.read_parquet("../data/features/data.parquet").filter(pl.col("ret_30min").is_not_null())

In [4]:
train_df = df.filter(pl.col("Time") < datetime(2019, 1, 1))
val_df = df.filter(
    (pl.col("Time") >= datetime(2019, 1, 1)) & (pl.col("Time") < datetime(2020, 1, 1))
)

In [5]:
features = [f for f in df.columns if "feature" in f]
target = "ret_30min"

In [6]:
X_train = train_df.select(features).to_numpy()
y_train = train_df.select(target).to_numpy().flatten()
X_val = val_df.select(features).to_numpy()
y_val = val_df.select(target).to_numpy().flatten()

In [7]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

In [8]:
model = LinearRegression()
model.fit(X_train_s, y_train)
y_pred_train = model.predict(X_train_s)
y_pred_val = model.predict(X_val_s)

In [21]:
ic_train = (y_pred_train * y_train).mean() / np.sqrt((y_pred_train ** 2).mean() * (y_train ** 2).mean())
ic_val = (y_pred_val * y_val).mean() / np.sqrt((y_pred_val ** 2).mean() * (y_val ** 2).mean())
print("ic_train:", ic_train)
print("ic_val:", ic_val)

ic_train: 0.059786161922411936
ic_val: 0.03795554157784214
